# GenAI Task 2: Deep Technical Blog on LangChain
**Internship:** Data Science Internship – February 2026 | Innomatics Research Labs  
**Companion Blog:** Published on Medium  

---

This notebook contains all working code examples from the LangChain blog.  
Each section matches a component covered in the blog.

### Architecture Flow
```
User Input → Prompt Template → Memory → Chain → Agent → LLM → Output Parser → Response
```

> **API Key Note:** This notebook uses OpenAI. Replace `your-openai-api-key-here` with your actual key,  
> or set it as an environment variable: `os.environ['OPENAI_API_KEY'] = 'sk-...'`  
> HuggingFace alternatives are included where possible (free tier).

---
## Cell 1 — Install All Dependencies

In [ ]:
# Install LangChain ecosystem packages
# langchain-core      : Core abstractions (prompts, runnables, output parsers)
# langchain           : Main chain and agent logic
# langchain-openai    : OpenAI LLM and embeddings integration
# langchain-community : Community integrations (HuggingFace, FAISS, loaders)
# faiss-cpu           : Vector store for similarity search
# openai              : OpenAI Python SDK

!pip install langchain langchain-core langchain-openai langchain-community faiss-cpu openai --quiet
!pip install huggingface_hub --quiet

print("All packages installed successfully.")

---
## Cell 2 — Imports and API Key Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# ── Set your API key here ─────────────────────────────────────────
# Option A: Set directly (replace with your actual key)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# Option B: Set HuggingFace token (free alternative)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your-hf-token-here"

print("Environment configured.")
print("Note: Replace placeholder keys with your actual API keys before running.")

---
## Cell 3 — Component 1: LLM and Chat Model

LangChain wraps different LLM providers behind a unified interface.  
You can swap providers without changing the rest of your code.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize OpenAI chat model
# temperature=0.7 → balanced between focused and creative
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7
)

# Basic invocation — pass a list of messages
messages = [
    SystemMessage(content="You are a concise technical assistant."),
    HumanMessage(content="What is LangChain in one sentence?")
]

response = llm.invoke(messages)

print("=" * 50)
print("LLM Response:")
print("-" * 50)
print(response.content)
print(f"\nModel     : {response.response_metadata.get('model_name', 'N/A')}")
print(f"Tokens used: {response.usage_metadata}")

In [ ]:
# ── HuggingFace Alternative (Free) ───────────────────────────────
# Use this if you don't have an OpenAI key

from langchain_community.llms import HuggingFaceHub

llm_hf = HuggingFaceHub(
    repo_id="google/flan-t5-base",          # Free model on HuggingFace
    model_kwargs={"temperature": 0.5, "max_length": 150}
)

hf_response = llm_hf.invoke("What is LangChain? Answer in one sentence.")

print("HuggingFace LLM Response:")
print(hf_response)

---
## Cell 4 — Component 2: Prompt Templates

Templates define a reusable prompt structure.  
Variables get filled in at runtime — no more string concatenation.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# ── Basic PromptTemplate ──────────────────────────────────────────
explain_template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="""Explain {topic} clearly for a {audience}.
Keep it under 100 words and use a simple analogy."""
)

# Format with actual values
prompt_1 = explain_template.format(topic="vector embeddings", audience="beginner")
prompt_2 = explain_template.format(topic="attention mechanism", audience="software engineer")

print("Formatted Prompt 1:")
print(prompt_1)
print("\nFormatted Prompt 2:")
print(prompt_2)

In [ ]:
# ── ChatPromptTemplate — for multi-turn conversation models ───────
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {domain} expert. Be concise and technical."),
    ("human", "Question: {question}"),
    ("ai", "I'll answer that clearly."),
    ("human", "Follow-up: {followup}")
])

# Format the full message chain
formatted_chat = chat_template.format_messages(
    domain="machine learning",
    question="What is gradient descent?",
    followup="Why do we use learning rate?"
)

print("ChatPromptTemplate Output:")
for msg in formatted_chat:
    print(f"  [{msg.type.upper()}]: {msg.content}")

---
## Cell 5 — Component 3: Chains (LCEL)

Chains connect components in a pipeline.  
LangChain Expression Language (LCEL) uses `|` to pipe data between steps.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)

# Define each component separately
summary_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Write a 3-sentence technical summary of: {topic}"
)

# StrOutputParser extracts just the text from the LLM response object
output_parser = StrOutputParser()

# ── Build chain using pipe operator ──────────────────────────────
# Flow: prompt → LLM → parser
summary_chain = summary_prompt | llm | output_parser

# Run the chain
result = summary_chain.invoke({"topic": "Retrieval Augmented Generation (RAG)"})

print("Chain Output:")
print("-" * 50)
print(result)

In [ ]:
# ── Sequential Chain: two LLM calls in sequence ──────────────────
# Step 1: Generate a concept explanation
# Step 2: Generate a quiz question about that explanation

explain_prompt = PromptTemplate(
    input_variables=["concept"],
    template="Explain {concept} in 2 sentences for a beginner."
)

quiz_prompt = PromptTemplate(
    input_variables=["explanation"],
    template="Based on this explanation: '{explanation}'\nWrite one multiple-choice quiz question with 4 options."
)

# Chain them: explanation output feeds into quiz prompt
full_chain = explain_prompt | llm | output_parser | quiz_prompt | llm | output_parser

quiz_result = full_chain.invoke({"concept": "neural networks"})

print("Sequential Chain — Quiz Generator:")
print("-" * 50)
print(quiz_result)

---
## Cell 6 — Component 4: Memory

LLMs have no built-in memory.  
LangChain memory modules inject conversation history into each prompt automatically.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)

# ConversationBufferMemory stores every turn verbatim
# Good for short conversations; gets expensive for long ones
memory = ConversationBufferMemory()

# ConversationChain handles injecting history into each call
chat = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# Simulated multi-turn conversation
print("=" * 55)
print("MEMORY DEMO — Multi-turn Conversation")
print("=" * 55)

turns = [
    "My name is Siddharth and I'm learning about LangChain.",
    "What am I learning about?",
    "What is my name?"
]

for i, turn in enumerate(turns, 1):
    response = chat.predict(input=turn)
    print(f"\nTurn {i}")
    print(f"  User: {turn}")
    print(f"  Bot : {response}")

# Show what's stored in memory
print("\n" + "-" * 55)
print("Stored Memory:")
mem_contents = memory.load_memory_variables({})
print(mem_contents['history'])

---
## Cell 7 — Component 5: Agents and Tools

Agents decide at runtime which tools to use.  
They follow the ReAct pattern: Reason → Act → Observe → Repeat.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_core.tools import Tool

# ── Define custom tools ───────────────────────────────────────────

def safe_calculator(expression: str) -> str:
    """Evaluates a math expression. Input: a valid expression like '25 * 4'."""
    try:
        # Restricted eval — only allows math, no builtins
        result = eval(expression, {"__builtins__": {}})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"


def text_word_counter(text: str) -> str:
    """Counts the number of words in a given text."""
    word_count = len(text.split())
    return f"Word count: {word_count} words"


# Wrap functions as LangChain Tools
tools = [
    Tool(
        name="Calculator",
        func=safe_calculator,
        description="Use for math calculations. Input must be a math expression like '15 * 0.18' or '(100 + 200) / 3'."
    ),
    Tool(
        name="WordCounter",
        func=text_word_counter,
        description="Use to count words in a piece of text. Input is the text string."
    )
]

# ── Create the ReAct agent ────────────────────────────────────────
llm_agent = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Pull the standard ReAct prompt from LangChain Hub
react_prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm=llm_agent, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,        # Shows Thought → Action → Observation steps
    max_iterations=5
)

# Test queries that require tool use
print("=" * 55)
print("AGENT DEMO")
print("=" * 55)

query = "What is 18% of 4500?"
print(f"Query: {query}\n")
result = agent_executor.invoke({"input": query})
print(f"\nFinal Answer: {result['output']}")

---
## Cell 8 — Component 6: Document Loaders + Vector Store (RAG Pipeline)

RAG = Retrieval Augmented Generation.  
Load documents → embed them → retrieve relevant chunks → answer questions.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA
from langchain_core.documents import Document

# ── Step 1: Create sample documents (simulates loading a real file) ───
# In production: use TextLoader, PyPDFLoader, WebBaseLoader etc.
raw_docs = [
    Document(page_content="""LangChain is a framework for building applications powered by large language models.
    It provides components for prompt management, memory, chains, and agents.
    LangChain supports OpenAI, HuggingFace, Anthropic, and many other LLM providers."""),

    Document(page_content="""Retrieval Augmented Generation (RAG) is a technique that combines information retrieval
    with text generation. The model retrieves relevant documents from a knowledge base before generating
    a response. This allows LLMs to answer questions about private or recent data without retraining."""),

    Document(page_content="""Vector embeddings are numerical representations of text in high-dimensional space.
    Similar texts have similar embeddings (small cosine distance). FAISS is a library by Meta for efficient
    similarity search over large collections of embeddings. It enables fast nearest-neighbor lookups."""),

    Document(page_content="""LangChain Expression Language (LCEL) uses the pipe operator | to chain components.
    Each component receives the output of the previous step as its input.
    LCEL supports streaming, async execution, and parallel branching out of the box.""")
]

# ── Step 2: Split documents into smaller chunks ────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,     # Max characters per chunk
    chunk_overlap=30    # Overlap preserves context at chunk boundaries
)
chunks = splitter.split_documents(raw_docs)
print(f"Original docs : {len(raw_docs)}")
print(f"After splitting: {len(chunks)} chunks")

# ── Step 3: Embed and store in FAISS vector store ──────────────────
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store created with FAISS.")

# ── Step 4: Build RetrievalQA chain ───────────────────────────────
llm_rag = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm_rag,
    # Retrieve top 2 most relevant chunks
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True
)

# ── Step 5: Ask questions ─────────────────────────────────────────
questions = [
    "What is RAG and why is it useful?",
    "What is LCEL and how does it work?"
]

print("\n" + "=" * 55)
print("RAG PIPELINE DEMO")
print("=" * 55)

for q in questions:
    print(f"\nQuestion: {q}")
    result = rag_chain.invoke({"query": q})
    print(f"Answer  : {result['result']}")
    print(f"Source chunks used: {len(result['source_documents'])}")

---
## Cell 9 — Summary: All Components Recap

| Component | Class Used | What It Does |
|---|---|---|
| LLM | `ChatOpenAI`, `HuggingFaceHub` | Generates text responses |
| Prompt Template | `PromptTemplate`, `ChatPromptTemplate` | Structures input with variables |
| Chain (LCEL) | `prompt \| llm \| parser` | Pipes data through components |
| Memory | `ConversationBufferMemory` | Stores and injects chat history |
| Agent | `create_react_agent` | Decides which tools to call |
| Tools | `Tool` | Custom functions the agent can use |
| Document Loader | `TextLoader`, `Document` | Loads text into LangChain |
| Vector Store | `FAISS` | Stores and retrieves embeddings |
| Output Parser | `StrOutputParser` | Cleans up LLM response format |

### Full RAG Architecture
```
Document (PDF/TXT/Web)
       ↓
Document Loader
       ↓
Text Splitter → Chunks
       ↓
Embeddings Model → Vectors
       ↓
FAISS Vector Store
       ↓ (at query time)
User Question → Retriever → Top-K Relevant Chunks
       ↓
Prompt Template (Question + Chunks)
       ↓
LLM → Answer
```

---
*Task 2 — GenAI Internship | Innomatics Research Labs | February 2026*  
*Portfolio: siddharthkaulagi.vercel.app*